In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import ast
import json
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.preprocessing import MultiLabelBinarizer
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score, roc_auc_score

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("neelshah18/arxivdataset")

Using Colab cache for faster access to the 'arxivdataset' dataset.


In [ ]:
base_path = '/kaggle/input/arxivdataset/arxivData.json'
df = pd.read_json(base_path)

In [ ]:
ARCHIVE_TAGS = {
    "cs", "econ", "eess", "math", "q-bio",
    "q-fin", "stat", "physics", "astro", "cond",
    "gr", "hep", "nlin", "nucl",
}

def parse_tag(value):
    if not isinstance(value, str):
        return value
    try:
        return json.loads(value.replace("'", '"'))
    except json.JSONDecodeError:
        return ast.literal_eval(value)

if isinstance(df["tag"].iloc[0], str):
    df["tag"] = df["tag"].apply(parse_tag)

unique_arxiv_terms = sorted({
    item["term"]
    for tag_list in df["tag"]
    for item in tag_list
    if ";" not in item["term"]
    and item["term"].split(".")[0] in ARCHIVE_TAGS
})

first_part_sections = sorted({
    term.split(".")[0] for term in unique_arxiv_terms
})

In [ ]:
def parse_tag_list(value):
    if isinstance(value, str):
        try:
            return ast.literal_eval(value)
        except (ValueError, SyntaxError):
            return []
    return value if isinstance(value, list) else []


def extract_categories(tag_list):
    tag_list = parse_tag_list(tag_list)

    return sorted({
        tag["term"].split(".")[0]
        for tag in tag_list
        if isinstance(tag, dict)
        and "term" in tag
        and tag["term"].split(".")[0] in ARCHIVE_TAGS
    })

df["categories"] = df["tag"].apply(extract_categories)

In [ ]:
df[['title', 'summary', 'categories']].head(10)

# Как видно, теги работают

,title,summary,categories
0,Dual Recurrent Attention Units for Visual Ques...,We propose an architecture for VQA which utili...,"[cs, stat]"
1,Sequential Short-Text Classification with Recu...,Recent approaches based on artificial neural n...,"[cs, stat]"
2,Multiresolution Recurrent Neural Networks: An ...,We introduce the multiresolution recurrent neu...,"[cs, stat]"
3,Learning what to share between loosely related...,Multi-task learning is motivated by the observ...,"[cs, stat]"
4,A Deep Reinforcement Learning Chatbot,We present MILABOT: a deep reinforcement learn...,"[cs, stat]"
5,Generating Sentences by Editing Prototypes,We propose a new generative model of sentences...,"[cs, stat]"
6,A Deep Reinforcement Learning Chatbot (Short V...,We present MILABOT: a deep reinforcement learn...,"[cs, stat]"
7,Document Image Coding and Clustering for Scrip...,The paper introduces a new method for discrimi...,[cs]
8,Tutorial on Answering Questions about Images w...,Together with the development of more accurate...,[cs]
9,pix2code: Generating Code from a Graphical Use...,Transforming a graphical user interface screen...,[cs]


In [ ]:
binarizer = MultiLabelBinarizer()
encoded_categories = binarizer.fit_transform(df["categories"])
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-cased")

def tokenize_sample(title, summary, max_length=256):
    return tokenizer(
        text=title,
        text_pair=summary,
        padding="max_length",
        truncation=True,
        max_length=max_length,
    )

tokenized_texts = [
    tokenize_sample(
        title=row["title"] if isinstance(row["title"], str) else "",
        summary=row["summary"] if isinstance(row["summary"], str) else "",
    )
    for _, row in df[["title", "summary"]].iterrows()
]


class CustomDataset(Dataset):
    def __init__(self, tokenized_data, targets):
        self.tokenized_data = tokenized_data
        self.targets = targets

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, index):
        item = self.tokenized_data[index]

        return {
            "input_ids": torch.tensor(item["input_ids"], dtype=torch.long),
            "attention_mask": torch.tensor(item["attention_mask"], dtype=torch.long),
            "labels": torch.tensor(self.targets[index], dtype=torch.float),
        }


dataset = CustomDataset(tokenized_texts, encoded_categories)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/465 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
train_indices, test_indices = train_test_split(
    range(len(dataset)),
    train_size=0.8,
    test_size=0.2,
    random_state=42
)

training_data = Subset(
    dataset=dataset,
    indices=train_indices
)

validation_data = Subset(
    dataset=dataset,
    indices=test_indices
)

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    'distilbert-base-cased',
    num_labels=9,
    problem_type='multi_label_classification'
)

training_args = TrainingArguments(
    output_dir='/content',
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    learning_rate=1e-5,
    fp16=True,
    dataloader_pin_memory=True,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=1,
    save_only_model=True,
    report_to=[]
)

def metrics_calc(model_outputs):
    raw_predictions, true_labels = model_outputs
    predictions = raw_predictions.flatten()
    targets = true_labels.astype(np.int32).flatten()
    roc_auc = roc_auc_score(y_true=targets, y_score=predictions)
    avg_precision = average_precision_score(y_true=targets, y_score=predictions)
    return {
        'roc_auc_score': roc_auc,
        'average_precision': avg_precision
    }

model.safetensors:   0%|          | 0.00/263M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-cased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [12]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)
model.to(device)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=training_data,
    eval_dataset=validation_data,
    compute_metrics=metrics_calc,
)

trainer.train()

cuda


Epoch,Training Loss,Validation Loss


Epoch,Training Loss,Validation Loss,Roc Auc Score,Average Precision
1,0.094423,0.093647,0.985437,0.949880
2,0.088498,0.089630,0.987565,0.953273
3,0.077891,0.088989,0.988020,0.954072
4,0.071179,0.089387,0.988385,0.954683
5,0.067854,0.090209,0.988180,0.953956


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=10250, training_loss=0.08535763289288777, metrics={'train_runtime': 1226.2262, 'train_samples_per_second': 133.744, 'train_steps_per_second': 8.359, 'total_flos': 1.0863682689024e+16, 'train_loss': 0.08535763289288777, 'epoch': 5.0})

In [18]:
torch.save(trainer.model.state_dict(), "model_weights.pth")

---

In [3]:
# !pip install streamlit

In [ ]:
!streamlit run app.py --server.port 8080




  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8080
  Network URL: http://10.33.0.2:8080
  External URL: http://103.31.77.165:8080

  For better performance, install the Watchdog module:

  $ xcode-select --install
  $ pip install watchdog
            
